# sources_test

Evaluate the **current ranking implementation** against the labeled dataset produced by `eval_dataset_builder.ipynb`.

This notebook:
- Loads a labeled dataset (by default the most recent under `eval_dataset/datasets/*/labeled_dataset.csv`; override via `DATASET_PATH`; falls back to `eval_dataset/labeled_dataset.csv`)
- Computes IR metrics per chapter (each chapter = one query)
- Logs results persistently to `eval_dataset/experiments/`

Important: labels are LLM-generated, so treat this as a **directional** benchmark. We’ll still evaluate across **3 different chapters** to reduce chapter-specific overfitting.


In [1]:
from __future__ import annotations

import os
import json
import hashlib
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Dataset selection
# - If `DATASET_PATH` env var is set, we use it.
# - Else we prefer the most-recent dataset-versioned build under `eval_dataset/datasets/*/labeled_dataset.csv`.
# - Else we fall back to the legacy path `eval_dataset/labeled_dataset.csv`.
DATASET_PATH_ENV = os.getenv("DATASET_PATH", "").strip()
DATASET_PATH = Path(DATASET_PATH_ENV) if DATASET_PATH_ENV else None
if DATASET_PATH is not None and not DATASET_PATH.exists():
    print(f"Warning: DATASET_PATH env var is set but missing: {DATASET_PATH}. Ignoring it.")
    DATASET_PATH = None
if DATASET_PATH is None:
    legacy = Path("eval_dataset/labeled_dataset.csv")
    datasets_root = Path("eval_dataset/datasets")
    candidates = []
    if datasets_root.exists():
        candidates = sorted(
            datasets_root.glob("*/labeled_dataset.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
    DATASET_PATH = candidates[0] if candidates else legacy

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing {DATASET_PATH}. Run eval_dataset_builder.ipynb first.")
print("Using DATASET_PATH:", DATASET_PATH)

EXP_DIR = Path("eval_dataset/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

KS = (10, 20, 50)
CONFIDENCE_MIN: Optional[int] = None  # e.g. 70 to filter low-confidence labels

EXPECTED_CHAPTERS = {
    "platform_theory",
    "platform_methodology",
    "platform_empirical_case",
}

# -----------------------------
# Run mode
# -----------------------------
# Choose what this notebook should do when you click "Run all".
# - "stageB_ab": evaluate BOTH datasets (baseline vs coverage_v1) and compare.
# - "single": evaluate DATASET_PATH once (classic mode).
# - "sweeps": run parameter sweeps (Stage C tuning) on the pinned dataset.
# - "rerank": run Stage C.3 LLM rerank tests (calls OpenAI).
# - "stageB_diag": run blueprint diagnostics (can call OpenAI).
RUN_MODE = "rerank"

# Stage B A/B settings (end-to-end)
STAGEB_AB_DATASET_TAGS = ["stageB_baseline_v1", "stageB_coverage_v1_v1"]
STAGEB_AB_EXPERIMENT_TAG = "stageB_ab_eval"
STAGEB_AB_SCORE_COL = "score_hybrid_pool"

if RUN_MODE == "stageB_ab":
    datasets_root = Path("eval_dataset/datasets")
    for tag in STAGEB_AB_DATASET_TAGS:
        p = datasets_root / tag / "labeled_dataset.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing dataset for tag '{tag}': {p}")

# Pinned dataset for Stage C tests (keeps results reproducible)
PINNED_DATASET_TAG = "stageB_coverage_v1_v1"  # change if you want to test a different dataset
if RUN_MODE in ("sweeps", "rerank"):
    datasets_root = Path("eval_dataset/datasets")
    p = datasets_root / PINNED_DATASET_TAG / "labeled_dataset.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing pinned dataset for tag '{PINNED_DATASET_TAG}': {p}")
    DATASET_PATH = p
    print("Pinned DATASET_PATH:", DATASET_PATH)

    def _get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"

    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    experiment_tag = "stageC_sweeps" if RUN_MODE == "sweeps" else "stageC3_rerank_v1"
    meta = {
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": _get_git_head(),
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": experiment_tag,
        "experiment_notes": f"pinned_dataset_tag={PINNED_DATASET_TAG}",
    }
    print("Pinned run_id:", run_id)

EXPERIMENT_TAG = (
    STAGEB_AB_EXPERIMENT_TAG
    if RUN_MODE == "stageB_ab"
    else ("stageC_sweeps" if RUN_MODE == "sweeps" else ("stageC3_rerank_v1" if RUN_MODE == "rerank" else "baseline"))
)
EXPERIMENT_NOTES = ""  # set automatically in stageB_ab mode

BASELINE_SCORE_COLS = [
    "score_hybrid_pool",
    "score_relevance_hybrid",
    "score_embed_combo",
    "score_tfidf",
    "score_stageC1",
    "score_cite_norm",
]

print("Config OK")
print("RUN_MODE:", RUN_MODE)
if RUN_MODE == "stageB_ab":
    print("Stage B A/B dataset tags:", STAGEB_AB_DATASET_TAGS)
if RUN_MODE in ("sweeps", "rerank"):
    print("Pinned dataset tag:", PINNED_DATASET_TAG)


Using DATASET_PATH: eval_dataset\datasets\stageB_coverage_v1_v1\labeled_dataset.csv
Pinned DATASET_PATH for sweeps: eval_dataset\datasets\stageB_coverage_v1_v1\labeled_dataset.csv
Sweeps run_id: 20260130_184909_7370e7a6e85f
Config OK
RUN_MODE: sweeps
Sweeps dataset tag: stageB_coverage_v1_v1


In [2]:
if RUN_MODE not in ("single", "sweeps", "rerank"):
    print(f"Skipping st_load (RUN_MODE={RUN_MODE})")
else:
    df = pd.read_csv(DATASET_PATH)
    print("Loaded:", DATASET_PATH, "shape=", df.shape)
    
    required = ["chapter_id", "merge_key", "final_label", "final_confidence"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Dataset missing required columns: {missing}")
    
    if CONFIDENCE_MIN is not None:
        df = df[df["final_confidence"].fillna(0) >= CONFIDENCE_MIN].copy()
        print(f"Filtered by final_confidence >= {CONFIDENCE_MIN}: shape=", df.shape)
    
    score_cols = [c for c in BASELINE_SCORE_COLS if c in df.columns]
    if not score_cols:
        raise RuntimeError("No score columns found to evaluate.")
    print("Score columns:", score_cols)
    
    print("\nChapters:")
    display(df.groupby("chapter_id").size().rename("n_docs").to_frame())
    
    present = set(df["chapter_id"].dropna().astype(str).unique().tolist())
    missing = sorted(EXPECTED_CHAPTERS - present)
    if missing:
        raise RuntimeError(f"Dataset is missing expected chapters: {missing}. Present: {sorted(present)}")
    
    print("\nLabel distribution:")
    display(df.groupby(["chapter_id", "final_label"]).size().unstack(fill_value=0))


Loaded: eval_dataset\datasets\stageB_coverage_v1_v1\labeled_dataset.csv shape= (660, 35)
Score columns: ['score_hybrid_pool', 'score_relevance_hybrid', 'score_embed_combo', 'score_tfidf', 'score_stageC1', 'score_cite_norm']

Chapters:


,n_docs
chapter_id,
platform_empirical_case,220
platform_methodology,220
platform_theory,220



Label distribution:


final_label,exclude,include,maybe
chapter_id,,,
platform_empirical_case,163,2,55
platform_methodology,114,2,104
platform_theory,100,19,101


In [3]:
LABEL_TO_BIN_INCLUDE = {"include": 1, "maybe": 0, "exclude": 0}
LABEL_TO_BIN_INCLUDE_OR_MAYBE = {"include": 1, "maybe": 1, "exclude": 0}
LABEL_TO_GRADE = {"include": 2, "maybe": 1, "exclude": 0}

def _to_numeric_score(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").fillna(-1e9).to_numpy(dtype=float)

def dcg(rels: np.ndarray) -> float:
    rels = np.asarray(rels, dtype=float)
    if rels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rels.size + 2))
    gains = (2.0 ** rels - 1.0)
    return float(np.sum(gains * discounts))

def ndcg_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = np.asarray(rels[:k], dtype=float)
    ideal = np.sort(rels)[::-1][:k]
    denom = dcg(ideal)
    if denom <= 0:
        return float("nan")
    return dcg(rels_k) / denom

def precision_at_k(rel_bin: np.ndarray, k: int) -> float:
    rel_bin = np.asarray(rel_bin[:k], dtype=float)
    return float(rel_bin.sum() / k) if k > 0 else float("nan")

def recall_at_k(rel_bin: np.ndarray, k: int) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    total = float(rel_bin.sum())
    if total <= 0:
        return float("nan")
    return float(rel_bin[:k].sum() / total)

def average_precision(rel_bin: np.ndarray) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    total = float(rel_bin.sum())
    if total <= 0:
        return float("nan")
    cumsum = np.cumsum(rel_bin)
    precision_at_i = cumsum / (np.arange(len(rel_bin)) + 1)
    return float((precision_at_i * rel_bin).sum() / total)

def mrr(rel_bin: np.ndarray) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    hits = np.where(rel_bin > 0)[0]
    if hits.size == 0:
        return 0.0
    return float(1.0 / (hits[0] + 1))

@dataclass(frozen=True)
class EvalResult:
    chapter_id: str
    score_col: str
    n_docs: int
    n_include: int
    n_maybe: int
    auc_include: float
    mrr_include: float
    p_at: Dict[int, float]
    r_at: Dict[int, float]
    ndcg_at: Dict[int, float]

def evaluate_chapter(g: pd.DataFrame, score_col: str, ks: Iterable[int]) -> EvalResult:
    scores = _to_numeric_score(g[score_col])
    order = np.argsort(-scores, kind="mergesort")

    labels = g["final_label"].fillna("exclude").astype(str).to_numpy()[order]
    rel_bin = np.array([LABEL_TO_BIN_INCLUDE.get(x, 0) for x in labels], dtype=float)
    rel_grade = np.array([LABEL_TO_GRADE.get(x, 0) for x in labels], dtype=float)

    # AUC for include vs non-include (only if both classes exist)
    try:
        auc = float(roc_auc_score(rel_bin, scores[order])) if len(set(rel_bin.tolist())) > 1 else float("nan")
    except Exception:
        auc = float("nan")

    p_at = {k: precision_at_k(rel_bin, k) for k in ks}
    r_at = {k: recall_at_k(rel_bin, k) for k in ks}
    ndcg = {k: ndcg_at_k(rel_grade, k) for k in ks}

    return EvalResult(
        chapter_id=str(g["chapter_id"].iloc[0]),
        score_col=score_col,
        n_docs=int(len(g)),
        n_include=int((g["final_label"] == "include").sum()),
        n_maybe=int((g["final_label"] == "maybe").sum()),
        auc_include=auc,
        mrr_include=mrr(rel_bin),
        p_at=p_at,
        r_at=r_at,
        ndcg_at=ndcg,
    )

def evaluate_all(df_in: pd.DataFrame, score_cols: List[str], ks: Iterable[int]) -> pd.DataFrame:
    chapters = sorted(df_in["chapter_id"].dropna().astype(str).unique().tolist())
    rows = []
    for score_col in score_cols:
        per_ch = []
        for cid in chapters:
            g = df_in[df_in["chapter_id"].astype(str) == cid].copy()
            r = evaluate_chapter(g, score_col=score_col, ks=ks)
            per_ch.append(r)

            row = {
                "chapter_id": r.chapter_id,
                "score_col": r.score_col,
                "n_docs": r.n_docs,
                "n_include": r.n_include,
                "n_maybe": r.n_maybe,
                "auc_include": r.auc_include,
                "mrr_include": r.mrr_include,
            }
            for k in ks:
                row[f"p@{k}"] = r.p_at[k]
                row[f"r@{k}"] = r.r_at[k]
                row[f"ndcg@{k}"] = r.ndcg_at[k]
            rows.append(row)

        # Macro average across chapters (each chapter = one query)
        macro = {
            "chapter_id": "__ALL__",
            "score_col": score_col,
            "n_docs": int(sum(x.n_docs for x in per_ch)),
            "n_include": int(sum(x.n_include for x in per_ch)),
            "n_maybe": int(sum(x.n_maybe for x in per_ch)),
            "auc_include": float(np.nanmean([x.auc_include for x in per_ch])),
            "mrr_include": float(np.nanmean([x.mrr_include for x in per_ch])),
        }
        for k in ks:
            macro[f"p@{k}"] = float(np.nanmean([x.p_at[k] for x in per_ch]))
            macro[f"r@{k}"] = float(np.nanmean([x.r_at[k] for x in per_ch]))
            macro[f"ndcg@{k}"] = float(np.nanmean([x.ndcg_at[k] for x in per_ch]))
        rows.append(macro)

    out = pd.DataFrame(rows)
    return out

print("Metrics utilities ready")


Metrics utilities ready


In [4]:
# Stage B end-to-end A/B runner (evaluates BOTH datasets and persists results)
#
# Safe to "Run all":
# - In RUN_MODE="stageB_ab" this is the main entrypoint.
# - It appends macro rows to `eval_dataset/experiments/runs.csv` for later comparison.

if RUN_MODE != "stageB_ab":
    print(f"Skipping st_stageB_ab_run (RUN_MODE={RUN_MODE})")
else:
    datasets_root = Path("eval_dataset/datasets")

    dataset_pairs: List[Tuple[str, Path]] = []
    for tag in STAGEB_AB_DATASET_TAGS:
        p = datasets_root / tag / "labeled_dataset.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing dataset for tag '{tag}': {p}")
        dataset_pairs.append((tag, p))

    def _get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"

    def _persist(results_df: pd.DataFrame, dataset_path: Path, experiment_tag: str, experiment_notes: str) -> None:
        dataset_sha = hashlib.sha1(dataset_path.read_bytes()).hexdigest()[:12]
        run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"

        meta = {
            "run_id": run_id,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "dataset_path": str(dataset_path),
            "dataset_sha1_12": dataset_sha,
            "git_head": _get_git_head(),
            "confidence_min": CONFIDENCE_MIN,
            "experiment_tag": experiment_tag,
            "experiment_notes": experiment_notes,
        }

        safe_tag = "".join([c if (c.isalnum() or c in "-_" ) else "_" for c in str(experiment_tag or "").strip()])
        safe_tag = safe_tag or "run"
        report_path = EXP_DIR / f"{run_id}_{safe_tag}_report.csv"
        results_df.assign(**meta).to_csv(report_path, index=False)
        print("Saved report:", report_path)

        runs_csv = EXP_DIR / "runs.csv"
        macro_rows = results_df[results_df["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
        macro_rows = macro_rows.assign(**meta, experiment=safe_tag)
        if runs_csv.exists():
            macro_rows.to_csv(runs_csv, mode="a", header=False, index=False)
        else:
            macro_rows.to_csv(runs_csv, index=False)
        print("Updated:", runs_csv)

    for tag, dataset_path in dataset_pairs:
        print(f"\n=== Evaluating dataset: {tag} ===")
        df_ab = pd.read_csv(dataset_path)

        required = ["chapter_id", "merge_key", "final_label", "final_confidence"]
        missing = [c for c in required if c not in df_ab.columns]
        if missing:
            raise RuntimeError(f"Dataset missing required columns {missing}: {dataset_path}")

        if CONFIDENCE_MIN is not None:
            df_ab = df_ab[df_ab["final_confidence"].fillna(0) >= CONFIDENCE_MIN].copy()
            print(f"Filtered by final_confidence >= {CONFIDENCE_MIN}: shape=", df_ab.shape)

        present = set(df_ab["chapter_id"].dropna().astype(str).unique().tolist())
        missing_ch = sorted(EXPECTED_CHAPTERS - present)
        if missing_ch:
            raise RuntimeError(f"Dataset is missing expected chapters: {missing_ch}. Present: {sorted(present)}")

        if STAGEB_AB_SCORE_COL not in df_ab.columns:
            raise RuntimeError(f"Missing required score column '{STAGEB_AB_SCORE_COL}' in {dataset_path}")

        res_ab = evaluate_all(df_ab, score_cols=[STAGEB_AB_SCORE_COL], ks=KS)
        print("Macro (__ALL__) summary:")
        display(res_ab[res_ab["chapter_id"] == "__ALL__"].reset_index(drop=True))

        _persist(
            res_ab,
            dataset_path=dataset_path,
            experiment_tag=STAGEB_AB_EXPERIMENT_TAG,
            experiment_notes=f"dataset_tag={tag}; score_col={STAGEB_AB_SCORE_COL}",
        )

    print("\nStage B A/B evaluation complete.")


Skipping st_stageB_ab_run (RUN_MODE=sweeps)


In [5]:
if RUN_MODE != "single":
    print(f"Skipping st_baseline_eval (RUN_MODE={RUN_MODE})")
else:
    results = evaluate_all(df, score_cols=score_cols, ks=KS)
    
    print("Macro (__ALL__) summary:")
    display(
        results[results["chapter_id"] == "__ALL__"]
        .sort_values(by=f"ndcg@{KS[1]}", ascending=False)
        .reset_index(drop=True)
    )
    
    print("Per-chapter detail:")
    display(
        results[results["chapter_id"] != "__ALL__"]
        .sort_values(["score_col", "chapter_id"], ascending=True)
        .reset_index(drop=True)
    )


Skipping st_baseline_eval (RUN_MODE=sweeps)


In [6]:
if RUN_MODE != "single":
    print(f"Skipping st_error_analysis (RUN_MODE={RUN_MODE})")
else:
    # Quick qualitative check: show high-ranked EXCLUDEs and low-ranked INCLUDEs
    
    def inspect_errors(df_in: pd.DataFrame, chapter_id: str, score_col: str, top_n: int = 8):
        g = df_in[df_in["chapter_id"] == chapter_id].copy()
        g["_score"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
        g = g.sort_values("_score", ascending=False).reset_index(drop=True)
    
        fp = g[g["final_label"] == "exclude"].head(top_n)
        fn = g[g["final_label"] == "include"].tail(top_n)
    
        cols = ["final_label", "final_confidence", score_col, "title", "venue", "year", "p1_short_reason", "p2_short_reason"]
        cols = [c for c in cols if c in g.columns]
    
        print(f"\n=== {chapter_id} | {score_col} ===")
        print("Top false positives (exclude but ranked high):")
        display(fp[cols])
        print("Bottom false negatives (include but ranked low):")
        display(fn[cols])
    
    best_score = (
        results[results["chapter_id"] == "__ALL__"]
        .sort_values(by=f"ndcg@{KS[1]}", ascending=False)
        .iloc[0]["score_col"]
    )
    
    for cid in sorted(df["chapter_id"].unique().tolist()):
        inspect_errors(df, chapter_id=cid, score_col=best_score, top_n=6)


Skipping st_error_analysis (RUN_MODE=sweeps)


In [7]:
if RUN_MODE != "single":
    print(f"Skipping st_persist (RUN_MODE={RUN_MODE})")
else:
    def get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"
    
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    git_head = get_git_head()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    
    meta = {
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": git_head,
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": EXPERIMENT_TAG,
        "experiment_notes": EXPERIMENT_NOTES,
    }
    
    safe_tag = "".join([c if (c.isalnum() or c in "-_" ) else "_" for c in str(EXPERIMENT_TAG or "").strip()])
    safe_tag = safe_tag or "run"
    report_path = EXP_DIR / f"{run_id}_{safe_tag}_report.csv"
    results.assign(**meta).to_csv(report_path, index=False)
    print("Saved report:", report_path)
    
    # Append macro rows to a persistent runs.csv for quick tracking
    runs_csv = EXP_DIR / "runs.csv"
    macro_rows = results[results["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
    macro_rows = macro_rows.assign(**meta, experiment=safe_tag)
    
    if runs_csv.exists():
        macro_rows.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        macro_rows.to_csv(runs_csv, index=False)
    
    print("Updated:", runs_csv)
    display(pd.read_csv(runs_csv).tail(25))


Skipping st_persist (RUN_MODE=sweeps)


In [8]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_cite_weight_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: citation-weight sweep (offline; no API calls)
# Goal: decide whether CITE_WEIGHT should be reduced/removed.

CITE_WEIGHTS = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.20]
BASE_COL = "score_relevance_hybrid"  # embed+tfidf hybrid (no citations)
CITE_COL = "score_cite_norm"
OUT_COL = "score_cite_sweep"

assert BASE_COL in df.columns, f"Missing {BASE_COL}"
assert CITE_COL in df.columns, f"Missing {CITE_COL}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_base_n"] = minmax_by_group(df_base, BASE_COL)
df_base["_cite_n"] = minmax_by_group(df_base, CITE_COL)

rows = []
for w in CITE_WEIGHTS:
    d = df_base.copy()
    d[OUT_COL] = (1.0 - float(w)) * d["_base_n"] + float(w) * d["_cite_n"]
    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "cite_weight": float(w),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["cite_weight"]) if not sweep_df.empty else 0.0
print(f"Best cite_weight by ndcg@20: {best_w:.3f}")

# Leave-one-chapter-out (LOCO) sanity check (with only 3 chapters)
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w in CITE_WEIGHTS:
        t = train.copy()
        t[OUT_COL] = (1.0 - float(w)) * t["_base_n"] + float(w) * t["_cite_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w)

    u = test.copy()
    u[OUT_COL] = (1.0 - float(best_train_w)) * u["_base_n"] + float(best_train_w) * u["_cite_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_cite_weight": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts (uses run_id/meta if available; otherwise creates local ids)
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "cite_weight_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_cite_weight_sweep.csv"
sweep_df.assign(**meta_use, experiment="cite_weight_sweep", base_col=BASE_COL, cite_col=CITE_COL).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_cite_weight_loco.csv"
loco_df.assign(**meta_use, experiment="cite_weight_loco", base_col=BASE_COL, cite_col=CITE_COL).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv (keeps history easy to scan)
d_best = df_base.copy()
d_best[OUT_COL] = (1.0 - best_w) * d_best["_base_n"] + best_w * d_best["_cite_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "cite_weight_sweep"
meta_best["experiment_notes"] = f"base={BASE_COL}; per-chapter minmax; best_w={best_w:.3f}"
best_macro = best_macro.assign(**meta_best, experiment="cite_weight_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


,cite_weight,ndcg@20,p@20,mrr_include,auc_include
0,0.08,0.636325,0.150000,0.671498,0.792585
1,0.20,0.634553,0.150000,0.670175,0.760300
2,0.12,0.633813,0.150000,0.670833,0.782426
3,0.05,0.628185,0.133333,0.671958,0.796736
4,0.00,0.596557,0.133333,0.505952,0.803945
5,0.01,0.595061,0.133333,0.505848,0.804032
6,0.02,0.594533,0.133333,0.505747,0.802154


Best cite_weight by ndcg@20: 0.080


,holdout_chapter,selected_cite_weight,train_ndcg@20,holdout_ndcg@20,holdout_p@20,holdout_mrr_include
0,platform_empirical_case,0.08,0.666405,0.576164,0.05,1.000000
1,platform_methodology,0.08,0.612797,0.683380,0.00,0.014493
2,platform_theory,0.20,0.636596,0.630466,0.40,1.000000


Avg LOCO holdout ndcg@20: 0.6300031216986884
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_cite_weight_sweep.csv
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_cite_weight_loco.csv
Appended best macro row to: eval_dataset\experiments\runs.csv


In [9]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_weight_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Example: offline sweep of a simple hybrid weight (no API calls)
# This is a safe, scientific starting point: change ONE thing, measure, log.

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

if ("score_embed_combo" in df.columns) and ("score_tfidf" in df.columns):
    df_sweep = df.copy()
    df_sweep["_emb_n"] = minmax_by_group(df_sweep, "score_embed_combo")
    df_sweep["_tf_n"] = minmax_by_group(df_sweep, "score_tfidf")
    cite = df_sweep["score_cite_norm"] if "score_cite_norm" in df_sweep.columns else 0.0

    rows = []
    for w in np.linspace(0, 1, 11):
        base = w * df_sweep["_emb_n"] + (1 - w) * df_sweep["_tf_n"]
        df_sweep["score_sweep"] = (1 - 0.08) * base + 0.08 * cite

        res_w = evaluate_all(df_sweep, score_cols=["score_sweep"], ks=KS)
        macro = res_w[res_w["chapter_id"] == "__ALL__"].iloc[0].to_dict()
        rows.append({
            "w_embed": float(w),
            "ndcg@20": float(macro.get("ndcg@20")),
            "p@20": float(macro.get("p@20")),
            "mrr_include": float(macro.get("mrr_include")),
        })

    sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
    display(sweep_df)

    best = sweep_df.iloc[0].to_dict()
    print("Best by ndcg@20:", best)

    # Save artifacts (ensure run_id/meta exist in sweeps mode)
    run_id_use = globals().get("run_id")
    meta_use = globals().get("meta")
    if run_id_use is None or meta_use is None:
        dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
        run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
        meta_use = {
            "run_id": run_id_use,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "dataset_path": str(DATASET_PATH),
            "dataset_sha1_12": dataset_sha,
            "git_head": "unknown",
            "confidence_min": CONFIDENCE_MIN,
            "experiment_tag": "weight_sweep",
            "experiment_notes": "auto-meta (run_id/meta missing)",
        }

    sweep_path = EXP_DIR / f"{run_id_use}_weight_sweep.csv"
    sweep_df.assign(**meta_use, experiment="weight_sweep").to_csv(sweep_path, index=False)
    print("Saved sweep:", sweep_path)
else:
    print("Skipping sweep: missing score_embed_combo or score_tfidf columns")
    '''
    exec(_CODE, globals())


,w_embed,ndcg@20,p@20,mrr_include
0,0.6,0.633892,0.150000,0.671429
1,0.5,0.628185,0.133333,0.671795
2,0.1,0.623527,0.133333,0.673469
3,0.3,0.619088,0.116667,0.672316
4,0.2,0.616043,0.133333,0.672956
5,0.4,0.615664,0.116667,0.672131
6,0.0,0.608700,0.116667,0.674419
7,0.7,0.602889,0.150000,0.504831
8,0.9,0.577207,0.150000,0.363541
9,0.8,0.560751,0.150000,0.237963


Best by ndcg@20: {'w_embed': 0.6000000000000001, 'ndcg@20': 0.6338917297135972, 'p@20': 0.15, 'mrr_include': 0.6714285714285714}
Saved sweep: eval_dataset\experiments\20260130_184909_7370e7a6e85f_weight_sweep.csv


In [10]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_embed_combo_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: embedding combo sweep (max vs breadth)
# Goal: tune how we combine score_embed_max vs score_embed_mean_top3.
# This stays chapter-agnostic and uses only already-computed columns (no API calls).

W_EMBED = 0.60  # from weight sweep best
W_TFIDF = 1.0 - W_EMBED

EMBED_MAX_WEIGHTS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
OUT_COL = "score_embed_combo_sweep"

for c in ["score_embed_max", "score_embed_mean_top3", "score_tfidf"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")

rows = []
for w_max in EMBED_MAX_WEIGHTS:
    d = df_base.copy()
    emb_max = pd.to_numeric(d["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(d["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    d["_emb_raw"] = float(w_max) * emb_max + (1.0 - float(w_max)) * emb_b
    d["_emb_n"] = minmax_by_group(d, "_emb_raw")

    d[OUT_COL] = W_EMBED * d["_emb_n"] + W_TFIDF * d["_tf_n"]

    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "w_embed_max": float(w_max),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_wmax = float(sweep_df.iloc[0]["w_embed_max"]) if not sweep_df.empty else 1.0
print(f"Best w_embed_max by ndcg@20: {best_wmax:.3f}")

# Leave-one-chapter-out (LOCO) sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w_max in EMBED_MAX_WEIGHTS:
        t = train.copy()
        emb_max = pd.to_numeric(t["score_embed_max"], errors="coerce").fillna(0.0)
        emb_b = pd.to_numeric(t["score_embed_mean_top3"], errors="coerce").fillna(0.0)
        t["_emb_raw"] = float(w_max) * emb_max + (1.0 - float(w_max)) * emb_b
        t["_emb_n"] = minmax_by_group(t, "_emb_raw")
        t[OUT_COL] = W_EMBED * t["_emb_n"] + W_TFIDF * t["_tf_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w_max)

    u = test.copy()
    emb_max = pd.to_numeric(u["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(u["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    u["_emb_raw"] = float(best_train_w) * emb_max + (1.0 - float(best_train_w)) * emb_b
    u["_emb_n"] = minmax_by_group(u, "_emb_raw")
    u[OUT_COL] = W_EMBED * u["_emb_n"] + W_TFIDF * u["_tf_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_w_embed_max": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "embed_combo_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_embed_combo_sweep.csv"
sweep_df.assign(**meta_use, experiment="embed_combo_sweep", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_embed_combo_loco.csv"
loco_df.assign(**meta_use, experiment="embed_combo_loco", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
emb_max = pd.to_numeric(d_best["score_embed_max"], errors="coerce").fillna(0.0)
emb_b = pd.to_numeric(d_best["score_embed_mean_top3"], errors="coerce").fillna(0.0)
d_best["_emb_raw"] = float(best_wmax) * emb_max + (1.0 - float(best_wmax)) * emb_b
d_best["_emb_n"] = minmax_by_group(d_best, "_emb_raw")
d_best[OUT_COL] = W_EMBED * d_best["_emb_n"] + W_TFIDF * d_best["_tf_n"]

best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "embed_combo_sweep"
meta_best["experiment_notes"] = f"W_EMBED={W_EMBED:.2f}; w_embed_max={best_wmax:.3f}; per-chapter minmax(tfidf, emb_raw)"
best_macro = best_macro.assign(**meta_best, experiment="embed_combo_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


,w_embed_max,ndcg@20,p@20,mrr_include,auc_include
0,0.0,0.632042,0.116667,0.673333,0.831489
1,0.2,0.629998,0.116667,0.673203,0.823166
2,0.4,0.597169,0.116667,0.506289,0.810672
3,0.5,0.596982,0.133333,0.506173,0.805320
4,0.6,0.592638,0.133333,0.505848,0.797238
5,0.7,0.585636,0.133333,0.505848,0.795011
6,0.8,0.574064,0.133333,0.338798,0.789223
7,1.0,0.568617,0.133333,0.338542,0.781885


Best w_embed_max by ndcg@20: 0.000


,holdout_chapter,selected_w_embed_max,train_ndcg@20,holdout_ndcg@20,holdout_p@20,holdout_mrr_include
0,platform_empirical_case,0.5,0.666891,0.457163,0.05,0.50
1,platform_methodology,0.0,0.606373,0.683380,0.00,0.02
2,platform_theory,0.0,0.641788,0.612549,0.30,1.00


Avg LOCO holdout ndcg@20: 0.584364215849294
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_embed_combo_sweep.csv
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_embed_combo_loco.csv
Appended best macro row to: eval_dataset\experiments\runs.csv


In [11]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_w_embed_breadth_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: re-sweep W_EMBED after we learned "breadth" > "max"
# We fix w_embed_max=0.0 (only score_embed_mean_top3) and sweep W_EMBED vs TF-IDF.
# No API calls.

EMBED_MAX_WEIGHT_FIXED = 0.0
W_EMBED_GRID = [round(float(x), 2) for x in np.linspace(0, 1, 11)]
OUT_COL = "score_w_embed_breadth_sweep"

for c in ["score_embed_mean_top3", "score_tfidf"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")
df_base["_emb_raw"] = pd.to_numeric(df_base["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_base["_emb_n"] = minmax_by_group(df_base, "_emb_raw")

rows = []
for w_embed in W_EMBED_GRID:
    d = df_base.copy()
    d[OUT_COL] = float(w_embed) * d["_emb_n"] + (1.0 - float(w_embed)) * d["_tf_n"]

    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "w_embed": float(w_embed),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["w_embed"]) if not sweep_df.empty else 0.6
print(f"Best w_embed (breadth) by ndcg@20: {best_w:.3f}")

# LOCO sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w_embed in W_EMBED_GRID:
        t = train.copy()
        t[OUT_COL] = float(w_embed) * t["_emb_n"] + (1.0 - float(w_embed)) * t["_tf_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w_embed)

    u = test.copy()
    u[OUT_COL] = float(best_train_w) * u["_emb_n"] + (1.0 - float(best_train_w)) * u["_tf_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_w_embed": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "w_embed_breadth_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_w_embed_breadth_sweep.csv"
sweep_df.assign(**meta_use, experiment="w_embed_breadth_sweep", w_embed_max_fixed=EMBED_MAX_WEIGHT_FIXED).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_w_embed_breadth_loco.csv"
loco_df.assign(**meta_use, experiment="w_embed_breadth_loco", w_embed_max_fixed=EMBED_MAX_WEIGHT_FIXED).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
d_best[OUT_COL] = float(best_w) * d_best["_emb_n"] + (1.0 - float(best_w)) * d_best["_tf_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "w_embed_breadth_sweep"
meta_best["experiment_notes"] = f"w_embed_max_fixed={EMBED_MAX_WEIGHT_FIXED:.1f}; best_w_embed={best_w:.3f}; per-chapter minmax(tfidf, emb_mean_top3)"
best_macro = best_macro.assign(**meta_best, experiment="w_embed_breadth_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


,w_embed,ndcg@20,p@20,mrr_include,auc_include
0,0.6,0.632042,0.116667,0.673333,0.831489
1,0.5,0.624936,0.116667,0.673203,0.834592
2,0.4,0.624181,0.116667,0.673469,0.842785
3,0.2,0.621485,0.133333,0.673611,0.846000
4,0.1,0.619581,0.133333,0.673611,0.845871
5,0.3,0.615692,0.116667,0.673469,0.844688
6,0.0,0.602497,0.116667,0.674419,0.849936
7,0.7,0.597462,0.133333,0.450980,0.816373
8,0.8,0.597297,0.150000,0.451111,0.802960
9,1.0,0.582136,0.133333,0.191595,0.764325


Best w_embed (breadth) by ndcg@20: 0.600


,holdout_chapter,selected_w_embed,train_ndcg@20,holdout_ndcg@20,holdout_p@20,holdout_mrr_include
0,platform_empirical_case,0.8,0.661834,0.468222,0.05,0.333333
1,platform_methodology,0.6,0.606373,0.683380,0.00,0.020000
2,platform_theory,0.6,0.641788,0.612549,0.30,1.000000


Avg LOCO holdout ndcg@20: 0.588050591413957
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_w_embed_breadth_sweep.csv
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_w_embed_breadth_loco.csv
Appended best macro row to: eval_dataset\experiments\runs.csv


In [12]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_cite_weight_breadth_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: citation weight on top of the improved "breadth" base score
# We fix the base to: W_EMBED * emb_mean_top3 + W_TFIDF * tfidf (per-chapter minmax),
# then sweep cite_weight.

W_EMBED = 0.60
W_TFIDF = 1.0 - W_EMBED

CITE_WEIGHTS = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.20, 0.30]
OUT_COL = "score_breadth_cite_sweep"

for c in ["score_embed_mean_top3", "score_tfidf", "score_cite_norm"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

# Build the fixed base score (breadth embedding + tfidf)
df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")
df_base["_emb_raw"] = pd.to_numeric(df_base["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_base["_emb_n"] = minmax_by_group(df_base, "_emb_raw")
df_base["_base"] = float(W_EMBED) * df_base["_emb_n"] + float(W_TFIDF) * df_base["_tf_n"]
df_base["_cite_n"] = minmax_by_group(df_base, "score_cite_norm")

rows = []
for w in CITE_WEIGHTS:
    d = df_base.copy()
    d[OUT_COL] = (1.0 - float(w)) * d["_base"] + float(w) * d["_cite_n"]
    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "cite_weight": float(w),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["cite_weight"]) if not sweep_df.empty else 0.0
print(f"Best cite_weight (breadth base) by ndcg@20: {best_w:.3f}")

# LOCO sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w in CITE_WEIGHTS:
        t = train.copy()
        t[OUT_COL] = (1.0 - float(w)) * t["_base"] + float(w) * t["_cite_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w)

    u = test.copy()
    u[OUT_COL] = (1.0 - float(best_train_w)) * u["_base"] + float(best_train_w) * u["_cite_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_cite_weight": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "cite_weight_breadth_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_cite_weight_breadth_sweep.csv"
sweep_df.assign(**meta_use, experiment="cite_weight_breadth_sweep", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_cite_weight_breadth_loco.csv"
loco_df.assign(**meta_use, experiment="cite_weight_breadth_loco", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
d_best[OUT_COL] = (1.0 - float(best_w)) * d_best["_base"] + float(best_w) * d_best["_cite_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "cite_weight_breadth_sweep"
meta_best["experiment_notes"] = f"W_EMBED={W_EMBED:.2f}; breadth emb (mean_top3); per-chapter minmax; best_cite_weight={best_w:.3f}"
best_macro = best_macro.assign(**meta_best, experiment="cite_weight_breadth_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


,cite_weight,ndcg@20,p@20,mrr_include,auc_include
0,0.20,0.660186,0.166667,0.671296,0.785093
1,0.12,0.634149,0.150000,0.671642,0.810365
2,0.02,0.632911,0.116667,0.673077,0.829457
3,0.01,0.632613,0.116667,0.673203,0.830200
4,0.00,0.632042,0.116667,0.673333,0.831489
5,0.05,0.629298,0.133333,0.672619,0.824542
6,0.08,0.626811,0.133333,0.672316,0.822597
7,0.30,0.614786,0.150000,0.281843,0.734861


Best cite_weight (breadth base) by ndcg@20: 0.200


,holdout_chapter,selected_cite_weight,train_ndcg@20,holdout_ndcg@20,holdout_p@20,holdout_mrr_include
0,platform_empirical_case,0.12,0.663656,0.575135,0.05,1.000000
1,platform_methodology,0.20,0.648589,0.683380,0.00,0.013889
2,platform_theory,0.20,0.669399,0.641760,0.45,1.000000


Avg LOCO holdout ndcg@20: 0.6334252684792331
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_cite_weight_breadth_sweep.csv
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_cite_weight_breadth_loco.csv
Appended best macro row to: eval_dataset\experiments\runs.csv


In [13]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_stageC_grid_search_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C: joint grid search (offline; no API calls)
#
# Why:
# Sequential sweeps can miss interactions (w_embed_max ↔ w_embed ↔ cite_weight).
# This grid search selects a single, globally best scoring recipe on the *winning Stage B dataset*.
#
# Score recipe:
# 1) emb_raw = w_embed_max * score_embed_max + (1-w_embed_max) * score_embed_mean_top3
# 2) per-chapter minmax normalize: emb_n, tfidf_n, cite_n
# 3) base = w_embed * emb_n + (1-w_embed) * tfidf_n
# 4) score = (1-cite_weight) * base + cite_weight * cite_n

REQUIRED = ["score_embed_max", "score_embed_mean_top3", "score_tfidf", "score_cite_norm"]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns for Stage C grid search: {missing}")

# Grids (keep small; can expand later)
W_EMBED_MAX_GRID = [0.0, 0.2, 0.5, 0.8, 1.0]
W_EMBED_GRID = [round(float(x), 2) for x in np.linspace(0, 1, 11)]
CITE_WEIGHT_GRID = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12]

OUT_COL = "score_stageC_grid_v1"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

def score_df(df_in: pd.DataFrame, w_embed_max: float, w_embed: float, cite_weight: float) -> pd.DataFrame:
    d = df_in.copy()
    emb_max = pd.to_numeric(d["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(d["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    d["_emb_raw"] = float(w_embed_max) * emb_max + (1.0 - float(w_embed_max)) * emb_b
    d["_emb_n"] = minmax_by_group(d, "_emb_raw")
    d["_tf_n"] = minmax_by_group(d, "score_tfidf")
    d["_cite_n"] = minmax_by_group(d, "score_cite_norm")

    base = float(w_embed) * d["_emb_n"] + (1.0 - float(w_embed)) * d["_tf_n"]
    d[OUT_COL] = (1.0 - float(cite_weight)) * base + float(cite_weight) * d["_cite_n"]
    return d

rows = []
for wmax in W_EMBED_MAX_GRID:
    for w in W_EMBED_GRID:
        for cw in CITE_WEIGHT_GRID:
            d = score_df(df, w_embed_max=wmax, w_embed=w, cite_weight=cw)
            res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
            macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
            rows.append({
                "w_embed_max": float(wmax),
                "w_embed": float(w),
                "cite_weight": float(cw),
                "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
                "p@20": float(macro.get("p@20", float("nan"))),
                "mrr_include": float(macro.get("mrr_include", float("nan"))),
                "auc_include": float(macro.get("auc_include", float("nan"))),
            })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df.head(25))

best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best Stage C grid (by ndcg@20):", best)

# LOCO sanity check (with only 3 chapters)
chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df[df["chapter_id"] != holdout].copy()
    test = df[df["chapter_id"] == holdout].copy()

    best_train = None
    best_train_ndcg20 = -1e9
    for wmax in W_EMBED_MAX_GRID:
        for w in W_EMBED_GRID:
            for cw in CITE_WEIGHT_GRID:
                dtr = score_df(train, w_embed_max=wmax, w_embed=w, cite_weight=cw)
                r = evaluate_all(dtr, score_cols=[OUT_COL], ks=KS)
                nd = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
                if nd > best_train_ndcg20:
                    best_train_ndcg20 = nd
                    best_train = {"w_embed_max": float(wmax), "w_embed": float(w), "cite_weight": float(cw)}

    dte = score_df(test, **best_train)
    r2 = evaluate_all(dte, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()
    loco_rows.append({
        "holdout_chapter": holdout,
        **best_train,
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC_grid_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_stageC_grid_v1_sweep.csv"
sweep_df.assign(**meta_use, experiment="stageC_grid_v1_sweep").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC_grid_v1_loco.csv"
loco_df.assign(**meta_use, experiment="stageC_grid_v1_loco").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
if best:
    d_best = score_df(df, w_embed_max=float(best["w_embed_max"]), w_embed=float(best["w_embed"]), cite_weight=float(best["cite_weight"]))
    best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
    best_macro["score_col"] = OUT_COL
    meta_best = dict(meta_use)
    meta_best["experiment_tag"] = "stageC_grid_v1"
    meta_best["experiment_notes"] = f"best_by=ndcg@20; w_embed_max={best['w_embed_max']}; w_embed={best['w_embed']}; cite_weight={best['cite_weight']}"
    best_macro = best_macro.assign(**meta_best, experiment="stageC_grid_v1_best")

    runs_csv = EXP_DIR / "runs.csv"
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)

    print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


,w_embed_max,w_embed,cite_weight,ndcg@20,p@20,mrr_include,auc_include
0,0.0,0.7,0.08,0.657353,0.166667,0.672414,0.809621
1,0.2,0.7,0.12,0.650761,0.183333,0.671171,0.786074
2,0.2,0.7,0.08,0.649755,0.150000,0.672131,0.798066
3,0.0,0.8,0.12,0.648164,0.166667,0.671498,0.784919
4,0.0,0.7,0.12,0.647564,0.166667,0.671569,0.802217
5,0.5,0.7,0.12,0.644112,0.166667,0.670782,0.767377
6,0.2,0.8,0.12,0.640515,0.166667,0.670996,0.772774
7,0.5,0.7,0.08,0.637755,0.150000,0.671362,0.778276
8,0.8,0.5,0.08,0.635474,0.150000,0.671296,0.793088
9,0.5,0.6,0.08,0.634571,0.150000,0.671498,0.792278


Best Stage C grid (by ndcg@20): {'w_embed_max': 0.0, 'w_embed': 0.7, 'cite_weight': 0.08, 'ndcg@20': 0.6573534985002133, 'p@20': 0.16666666666666666, 'mrr_include': 0.6724137931034483, 'auc_include': 0.8096212163070051}


,holdout_chapter,w_embed_max,w_embed,cite_weight,train_ndcg@20,holdout_ndcg@20,holdout_p@20,holdout_mrr_include
0,platform_empirical_case,0.2,0.7,0.12,0.680850,0.590583,0.05,1.000000
1,platform_methodology,0.0,0.7,0.08,0.644340,0.683380,0.00,0.017241
2,platform_theory,0.2,0.7,0.08,0.653745,0.641775,0.40,1.000000


Avg LOCO holdout ndcg@20: 0.638579353865978
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_stageC_grid_v1_sweep.csv
Saved: eval_dataset\experiments\20260130_184909_7370e7a6e85f_stageC_grid_v1_loco.csv
Appended best macro row to: eval_dataset\experiments\runs.csv


In [ ]:
if RUN_MODE != "rerank":
    print(f"Skipping st_stageC3_llm_rerank_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: LLM rerank test (API calls; cached to disk)
#
# Scientific intent:
# - Keep Stage B fixed (coverage_v1 dataset).
# - Keep labels fixed (eval_rubrics).
# - Measure whether an LLM-based rerank score improves ranking quality.

import os
import json
import time
import random
import asyncio
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

from openai import OpenAI
from agents import Agent, Runner, ModelSettings


# -----------------------------
# Settings
# -----------------------------
RERANK_MODEL = os.getenv("RERANK_MODEL", "gpt-5-nano").strip() or "gpt-5-nano"
PROMPT_VERSION = "v1"
FORCE_RERANK = False  # set True to ignore cache

CONCURRENCY = 8
MAX_RETRIES = 8
BACKOFF_INITIAL = 1.0
BACKOFF_MAX = 30.0
ABSTRACT_MAX_CHARS = 2000

# Pricing for cost estimation (USD per 1M tokens)
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
}


def _price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    prices = _price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


def _truncate(s: str, max_chars: int) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + "…"


RUBRIC_DIR = Path("eval_dataset/eval_rubrics")
if not RUBRIC_DIR.exists():
    raise FileNotFoundError(f"Missing rubric dir: {RUBRIC_DIR}")

rubrics: Dict[str, dict] = {}
rubric_sigs: Dict[str, str] = {}
for cid in sorted(EXPECTED_CHAPTERS):
    rp = RUBRIC_DIR / f"{cid}.json"
    if not rp.exists():
        raise FileNotFoundError(f"Missing rubric for chapter '{cid}': {rp}")
    r = json.loads(rp.read_text(encoding="utf-8"))
    rubrics[cid] = r
    payload = {
        "scope_statement": r.get("scope_statement"),
        "must_cover": r.get("must_cover", []),
        "must_avoid": r.get("must_avoid", []),
        "scoring_guidance": r.get("scoring_guidance"),
    }
    rubric_sigs[cid] = hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


class RerankOut(BaseModel):
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)


rerank_agent = Agent(
    name=f"StageC3 Rerank ({PROMPT_VERSION})",
    model=RERANK_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You score academic papers for inclusion in a specific thesis chapter. "
        "Use the rubric strictly. Return ONLY the structured output."
    ),
    output_type=RerankOut,
)


def build_prompt(rubric: dict, title: str, abstract: str) -> str:
    scope = rubric.get("scope_statement", "")
    must_cover = rubric.get("must_cover", [])
    must_avoid = rubric.get("must_avoid", [])
    guidance = rubric.get("scoring_guidance", "")

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")

    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {title}")
    if abstract:
        lines.append(f"ABSTRACT: {abstract}")
    else:
        lines.append("ABSTRACT: (missing)")

    lines.append("")
    lines.append("Scoring:")
    lines.append("- 100 = must cite for this chapter")
    lines.append("- 50 = partially relevant / might cite")
    lines.append("- 0 = irrelevant or off-scope")
    return "\n".join(lines)


CACHE_DIR = EXP_DIR / "llm_rerank_cache_v1" / PINNED_DATASET_TAG
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_path(chapter_id: str, merge_key: str) -> Path:
    mk = str(merge_key)
    mk_h = hashlib.sha1(mk.encode("utf-8")).hexdigest()[:16]
    sig = rubric_sigs.get(str(chapter_id), "nosig")
    safe_model = RERANK_MODEL.replace("/", "_")
    d = CACHE_DIR / str(chapter_id)
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{mk_h}_{sig}_{PROMPT_VERSION}_{safe_model}.json"


def atomic_write_json(path: Path, obj: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


async def score_one(row: pd.Series) -> dict:
    cid = str(row["chapter_id"])
    mk = str(row["merge_key"])
    title = str(row.get("title", "") or "").strip()
    abstract = _truncate(str(row.get("abstract", "") or ""), ABSTRACT_MAX_CHARS)

    out_path = cache_path(cid, mk)
    if (not FORCE_RERANK) and out_path.exists():
        cached = json.loads(out_path.read_text(encoding="utf-8"))
        cached["_cached"] = True
        return cached

    rubric = rubrics[cid]
    prompt = build_prompt(rubric, title=title, abstract=abstract)

    last_err: Optional[str] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = await Runner.run(rerank_agent, prompt)
            out = res.final_output.model_dump()
            usage = res.context_wrapper.usage
            cost = cost_from_usage(usage, model=RERANK_MODEL)
            payload = {
                "chapter_id": cid,
                "merge_key": mk,
                **out,
                "_meta": {
                    "model": RERANK_MODEL,
                    "prompt_version": PROMPT_VERSION,
                    "rubric_sig": rubric_sigs.get(cid),
                    **cost,
                },
                "_cached": False,
            }
            atomic_write_json(out_path, payload)
            return payload
        except Exception as e:
            last_err = repr(e)
            backoff = min(BACKOFF_MAX, BACKOFF_INITIAL * (2 ** (attempt - 1)))
            backoff *= (1.0 + random.uniform(-0.15, 0.15))
            await asyncio.sleep(max(0.1, backoff))

    raise RuntimeError(f"Rerank failed after {MAX_RETRIES} retries for {cid} {mk}: {last_err}")


async def run_all() -> List[dict]:
    sem = asyncio.Semaphore(CONCURRENCY)
    rows = df.copy()

    async def _bound(row: pd.Series) -> dict:
        async with sem:
            return await score_one(row)

    tasks = [_bound(r) for _, r in rows.iterrows()]
    out: List[dict] = []
    for fut in asyncio.as_completed(tasks):
        out.append(await fut)
        if len(out) % 50 == 0:
            print(f"Scored {len(out)}/{len(tasks)}")
    return out


t0 = time.time()
client = OpenAI()  # ensure API client init
print("Starting rerank… model=", RERANK_MODEL, "docs=", len(df))
results = await run_all()
seconds = time.time() - t0

# Build score columns
scores = pd.DataFrame([
    {
        "chapter_id": r.get("chapter_id"),
        "merge_key": r.get("merge_key"),
        "llm_score": float(r.get("score", 0.0)),
        "llm_notes": r.get("notes", ""),
        "llm_cached": bool(r.get("_cached", False)),
        "cost_usd": float((r.get("_meta") or {}).get("cost_usd", 0.0)),
        "input_tokens": int((r.get("_meta") or {}).get("input_tokens", 0)),
        "cached_input_tokens": int((r.get("_meta") or {}).get("cached_input_tokens", 0)),
        "output_tokens": int((r.get("_meta") or {}).get("output_tokens", 0)),
        "requests": int((r.get("_meta") or {}).get("requests", 1)),
    }
    for r in results
])

totals = {
    "seconds": float(seconds),
    "requests": float(scores["requests"].sum()),
    "input_tokens": float(scores["input_tokens"].sum()),
    "cached_input_tokens": float(scores["cached_input_tokens"].sum()),
    "output_tokens": float(scores["output_tokens"].sum()),
    "cost_usd": float(scores["cost_usd"].sum()),
    "cached_files": int(scores["llm_cached"].sum()),
}
print("Rerank totals:", totals)

# Merge into dataset
df_r = df.merge(scores[["chapter_id", "merge_key", "llm_score", "llm_notes"]], on=["chapter_id", "merge_key"], how="left")
df_r["score_llm_rerank_v1"] = pd.to_numeric(df_r["llm_score"], errors="coerce").fillna(0.0) / 100.0

# Stage C finalized score (for comparison)
STAGEC_W_EMBED_MAX = 0.0
STAGEC_W_EMBED = 0.7
STAGEC_CITE_WEIGHT = 0.08

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

emb_max = pd.to_numeric(df_r["score_embed_max"], errors="coerce").fillna(0.0)
emb_b = pd.to_numeric(df_r["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_r["_emb_raw"] = float(STAGEC_W_EMBED_MAX) * emb_max + (1.0 - float(STAGEC_W_EMBED_MAX)) * emb_b
df_r["_emb_n"] = minmax_by_group(df_r, "_emb_raw")
df_r["_tf_n"] = minmax_by_group(df_r, "score_tfidf")
df_r["_cite_n"] = minmax_by_group(df_r, "score_cite_norm")
base = float(STAGEC_W_EMBED) * df_r["_emb_n"] + (1.0 - float(STAGEC_W_EMBED)) * df_r["_tf_n"]
df_r["score_stageC_final"] = (1.0 - float(STAGEC_CITE_WEIGHT)) * base + float(STAGEC_CITE_WEIGHT) * df_r["_cite_n"]

# Evaluate
score_cols = ["score_hybrid_pool", "score_stageC_final", "score_llm_rerank_v1"]
res = evaluate_all(df_r, score_cols=score_cols, ks=KS)

macro = res[res["chapter_id"] == "__ALL__"].sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(macro)

# Per-chapter ndcg@20 (quick sanity)
per_ndcg = res[res["chapter_id"] != "__ALL__"].pivot_table(index="chapter_id", columns="score_col", values="ndcg@20")
display(per_ndcg)

# Macro delta: LLM vs StageC_final
need = {"score_llm_rerank_v1", "score_stageC_final"}
if not macro.empty and need.issubset(set(macro["score_col"])):
    m = macro.set_index("score_col")
    cols = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in m.columns]
    delta = (m.loc["score_llm_rerank_v1", cols] - m.loc["score_stageC_final", cols]).to_frame("delta")
    print("Delta (score_llm_rerank_v1 - score_stageC_final):")
    display(delta)

# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

scored_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_scored.csv"
df_r.to_csv(scored_path, index=False)
print("Saved:", scored_path)

eval_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_eval.csv"
res.assign(**meta_use, experiment="stageC3_rerank_v1_eval").to_csv(eval_path, index=False)
print("Saved:", eval_path)

cost_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_cost.csv"
pd.DataFrame([totals]).assign(**meta_use, model=RERANK_MODEL, prompt_version=PROMPT_VERSION).to_csv(cost_path, index=False)
print("Saved:", cost_path)

# Append best macro row (LLM rerank) to runs.csv
best_macro = macro[macro["score_col"] == "score_llm_rerank_v1"].copy()
if not best_macro.empty:
    best_macro = best_macro.assign(**meta_use, experiment="stageC3_rerank_v1")
    runs_csv = EXP_DIR / "runs.csv"
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended LLM rerank macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


## Stage B — Blueprint tests (stability + quality)

These tests validate that blueprint generation is stable and that facet queries are diverse/useful.

- Some cells call the OpenAI API (cost-tracked).
- All results are saved under `eval_dataset/experiments/<stageB_run_id>_stageB/`.


In [14]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_config (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
import os
import re
import json
import time
import asyncio
import hashlib
import difflib
from itertools import combinations
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer

# API-only dependencies
try:
    from openai import OpenAI
    from agents import Agent, Runner, ModelSettings
    STAGEB_AGENTS_OK = True
except Exception as e:
    STAGEB_AGENTS_OK = False
    print("Warning: openai/agents not available -> API-based Stage B tests will be skipped.")
    print("Import error:", repr(e))

# -----------------------------
# Stage B config
# -----------------------------
DO_STAGEB_API_CALLS = True
STAGEB_FORCE_REGEN = False
STAGEB_CONCURRENCY = 5

BLUEPRINT_MODEL = "gpt-5-mini"
BLUEPRINT_RUNS_PER_CHAPTER = 5  # stability reps

# Pricing for cost estimates (update if pricing changes)
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
}

def _price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})

def cost_from_usage(usage, model: str) -> dict:
    prices = _price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)
        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)
    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }

# Local StageB run folder
dataset_sha12 = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
stageB_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha12}"
STAGEB_DIR = EXP_DIR / f"{stageB_run_id}_stageB"
STAGEB_DIR.mkdir(parents=True, exist_ok=True)
STABILITY_DIR = STAGEB_DIR / "stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS_DIR = STAGEB_DIR / "variants"
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)

# Load CHAPTERS from eval_dataset_builder.ipynb to avoid duplication/drift
nb = json.loads(Path("eval_dataset_builder.ipynb").read_text(encoding="utf-8"))
spec_cell = next(c for c in nb.get("cells", []) if c.get("id") == "eval_chapter_specs")
spec_src = "".join(spec_cell.get("source", []))
ns: Dict[str, Any] = {"List": List, "Dict": Dict, "Any": Any}
exec(spec_src, ns, ns)
CHAPTERS: List[Dict[str, Any]] = ns["CHAPTERS"]

# Blueprint output schema (same as eval_dataset_builder.ipynb)
class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")
    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]
    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]
    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None
    scoring_guidance: str
    notes: Optional[str] = None

BASE_BLUEPRINT_INSTRUCTIONS = (
    "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
    "Return ONLY the structured output fields (no extra text).\n\n"
    "Constraints:\n"
    "- language must be 'en'\n"
    "- scope_statement: 1 sentence, <= 30 words\n"
    "- must_cover: 4–8 bullets, each <= 16 words\n"
    "- should_cover: 3–8 bullets, each <= 16 words\n"
    "- must_avoid: 3–8 bullets, each <= 16 words\n"
    "- main_query: <= 18 words\n"
    "- facet_queries: 8–14 items, each <= 14 words\n"
    "- keywords: 20–45 items\n"
    "- key_concepts: 10–22 items\n"
    "- preferred_source_types: 2–6 items\n"
    "- negative_query_terms: 0–12 items derived from must_avoid (soft negatives)\n"
    "- scoring_guidance: <= 80 words\n"
    "- Do NOT contradict yourself: if something is in must_avoid, do not emphasize it in keywords/facets.\n"
    "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
)

VARIANT_COVERAGE_V1_INSTRUCTIONS = BASE_BLUEPRINT_INSTRUCTIONS + (
    "\nAdditional requirements (coverage_v1):\n"
    "- facet_queries must be semantically diverse (avoid near-duplicates).\n"
    "- Ensure each must_cover bullet is explicitly targeted by at least one facet_query.\n"
)

print("Stage B config OK")
print("- stageB_run_id:", stageB_run_id)
print("- STAGEB_DIR:", STAGEB_DIR)
print("- CHAPTERS:", [c["chapter_id"] for c in CHAPTERS])
    '''
    exec(_CODE, globals())


Skipping st_stageB_config (RUN_MODE=sweeps)


In [15]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_load_base_blueprints (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Load the current baseline blueprints that were used to build the dataset
BASE_BLUEPRINT_DIR = DATASET_PATH.parent / "blueprints"
if not BASE_BLUEPRINT_DIR.exists():
    # Backwards compatibility (older datasets)
    BASE_BLUEPRINT_DIR = Path("eval_dataset/blueprints")

assert BASE_BLUEPRINT_DIR.exists(), f"Missing {BASE_BLUEPRINT_DIR}. Run eval_dataset_builder.ipynb first."

base_blueprints: Dict[str, dict] = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    p = BASE_BLUEPRINT_DIR / f"{cid}.json"
    assert p.exists(), f"Missing blueprint: {p}"
    base_blueprints[cid] = json.loads(p.read_text(encoding="utf-8"))

rows = []
for cid, bp in base_blueprints.items():
    rows.append({
        "chapter_id": cid,
        "n_facets": len(bp.get("facet_queries", []) or []),
        "n_keywords": len(bp.get("keywords", []) or []),
        "n_key_concepts": len(bp.get("key_concepts", []) or []),
        "main_query": str(bp.get("main_query", ""))[:90],
    })

display(pd.DataFrame(rows))
print("Loaded baseline blueprints from:", BASE_BLUEPRINT_DIR)
    '''
    exec(_CODE, globals())


Skipping st_stageB_load_base_blueprints (RUN_MODE=sweeps)


In [16]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_blueprint_stability_runs (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 1 (must-do): Blueprint stability
# Generate multiple blueprints per chapter with identical settings and measure overlap.
# Cost: ~ (3 * BLUEPRINT_RUNS_PER_CHAPTER) calls to BLUEPRINT_MODEL.

if not DO_STAGEB_API_CALLS:
    print("Skipping blueprint stability runs (DO_STAGEB_API_CALLS=False)")
elif not STAGEB_AGENTS_OK:
    print("Skipping blueprint stability runs (openai/agents import failed)")
else:
    assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
    _ = OpenAI()  # ensure client init

    blueprint_agent = Agent(
        name="Chapter Blueprint Builder (stability)",
        model=BLUEPRINT_MODEL,
        model_settings=ModelSettings(top_p=1.0, verbosity="low"),
        instructions=BASE_BLUEPRINT_INSTRUCTIONS,
        output_type=ChapterBlueprint,
    )

    sem = asyncio.Semaphore(int(STAGEB_CONCURRENCY))

    async def gen_one(chapter: dict, rep_i: int) -> dict:
        out_path = STABILITY_DIR / f"{chapter['chapter_id']}__r{rep_i:02d}.json"
        if out_path.exists() and not STAGEB_FORCE_REGEN:
            return {"chapter_id": chapter["chapter_id"], "rep": rep_i, "path": str(out_path), "cached": True, **{"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}}

        prompt = (
            "Create a ChapterBlueprint for academic literature retrieval.\n"
            "Return ONLY the structured output fields required by the schema.\n\n"
            "CHAPTER_SPEC_JSON:\n" + json.dumps(chapter, ensure_ascii=False, indent=2)
        )

        async with sem:
            res = await Runner.run(blueprint_agent, prompt)

        bp = res.final_output.model_dump()
        bp["_meta"] = {
            "chapter_id": chapter["chapter_id"],
            "rep": int(rep_i),
            "model": BLUEPRINT_MODEL,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

        usage = res.context_wrapper.usage
        u = cost_from_usage(usage, model=BLUEPRINT_MODEL)
        return {"chapter_id": chapter["chapter_id"], "rep": rep_i, "path": str(out_path), "cached": False, **u}

    t0 = time.time()
    tasks = [gen_one(ch, rep_i) for ch in CHAPTERS for rep_i in range(1, int(BLUEPRINT_RUNS_PER_CHAPTER) + 1)]
    run_rows = await asyncio.gather(*tasks)
    dt = time.time() - t0

    runs_df = pd.DataFrame(run_rows)
    display(runs_df)

    totals = {k: float(runs_df[k].sum()) for k in ["requests", "input_tokens", "cached_input_tokens", "output_tokens", "cost_usd"] if k in runs_df.columns}
    totals["cached_files"] = int((runs_df.get("cached") == True).sum())
    totals["seconds"] = float(dt)
    print("Blueprint stability run totals:", totals)

    out_csv = STAGEB_DIR / "blueprint_stability_runs.csv"
    runs_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_blueprint_stability_runs (RUN_MODE=sweeps)


In [17]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_blueprint_stability_metrics (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Analyze stability runs: overlap of facets/keywords/etc across repeated blueprint generations.

def _norm_item(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def _token_set(s: str) -> set:
    return set(re.findall(r"[a-z0-9]+", (s or "").lower()))

def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    u = a | b
    return float(len(a & b) / len(u)) if u else float("nan")

def _avg_min(vals: List[float]) -> tuple[float, float]:
    if not vals:
        return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.min(vals))

def stability_list(bps: List[dict], field: str) -> tuple[float, float]:
    sets = [set(_norm_item(x) for x in (bp.get(field) or [])) for bp in bps]
    sims = [_jaccard(sets[i], sets[j]) for i, j in combinations(range(len(sets)), 2)]
    return _avg_min(sims)

def stability_text_tokens(bps: List[dict], field: str) -> tuple[float, float]:
    toks = [_token_set(bp.get(field) or "") for bp in bps]
    sims = [_jaccard(toks[i], toks[j]) for i, j in combinations(range(len(toks)), 2)]
    return _avg_min(sims)

def stability_text_seqratio(bps: List[dict], field: str) -> tuple[float, float]:
    texts = [_norm_item(bp.get(field) or "") for bp in bps]
    sims = [difflib.SequenceMatcher(None, texts[i], texts[j]).ratio() for i, j in combinations(range(len(texts)), 2)]
    return _avg_min(sims)

rows = []
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    files = sorted(STABILITY_DIR.glob(f"{cid}__r*.json"))
    if not files:
        print(f"No stability blueprints for {cid} in {STABILITY_DIR} (run the stability cell first)")
        continue

    bps = [json.loads(p.read_text(encoding="utf-8")) for p in files]
    hashes = [hashlib.sha1(p.read_bytes()).hexdigest() for p in files]

    facet_avg, facet_min = stability_list(bps, "facet_queries")
    kw_avg, kw_min = stability_list(bps, "keywords")
    kc_avg, kc_min = stability_list(bps, "key_concepts")
    mc_avg, mc_min = stability_list(bps, "must_cover")
    ma_avg, ma_min = stability_list(bps, "must_avoid")
    mq_avg, mq_min = stability_text_tokens(bps, "main_query")
    scope_avg, scope_min = stability_text_tokens(bps, "scope_statement")
    guide_avg, guide_min = stability_text_seqratio(bps, "scoring_guidance")

    rows.append({
        "stageB_run_id": stageB_run_id,
        "chapter_id": cid,
        "n_runs": int(len(files)),
        "n_unique_outputs": int(len(set(hashes))),
        "facet_jaccard_avg": facet_avg,
        "facet_jaccard_min": facet_min,
        "keywords_jaccard_avg": kw_avg,
        "keywords_jaccard_min": kw_min,
        "key_concepts_jaccard_avg": kc_avg,
        "key_concepts_jaccard_min": kc_min,
        "must_cover_jaccard_avg": mc_avg,
        "must_cover_jaccard_min": mc_min,
        "must_avoid_jaccard_avg": ma_avg,
        "must_avoid_jaccard_min": ma_min,
        "main_query_token_jaccard_avg": mq_avg,
        "main_query_token_jaccard_min": mq_min,
        "scope_token_jaccard_avg": scope_avg,
        "scope_token_jaccard_min": scope_min,
        "scoring_guidance_seqratio_avg": guide_avg,
        "scoring_guidance_seqratio_min": guide_min,
    })

stability_df = pd.DataFrame(rows)
display(stability_df)

out_csv = STAGEB_DIR / "blueprint_stability_metrics.csv"
stability_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_blueprint_stability_metrics (RUN_MODE=sweeps)


In [18]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_facet_redundancy (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 2 (must-do): Facet diversity / redundancy
# We compute TF-IDF cosine similarity across facet_queries (per blueprint) and flag near-duplicates.

def facet_redundancy_stats(bp: dict) -> dict:
    facets = [str(x) for x in (bp.get("facet_queries") or []) if str(x).strip()]
    main = str(bp.get("main_query") or "").strip()
    if len(facets) < 2:
        return {"n_facets": len(facets), "max_sim": float("nan"), "mean_sim": float("nan"), "pct_gt_0_8": float("nan"), "max_sim_to_main": float("nan")}

    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vec.fit_transform(facets)
    sim = (X @ X.T).toarray()  # TF-IDF is L2 normalized -> dot == cosine
    n = sim.shape[0]
    off = sim[~np.eye(n, dtype=bool)]

    qsim = float("nan")
    if main:
        q = vec.transform([main])
        qsim = float((q @ X.T).toarray().ravel().max()) if X.shape[0] else float("nan")

    return {
        "n_facets": int(len(facets)),
        "max_sim": float(np.max(off)) if off.size else float("nan"),
        "mean_sim": float(np.mean(off)) if off.size else float("nan"),
        "pct_gt_0_8": float(np.mean(off > 0.8)) if off.size else float("nan"),
        "pct_gt_0_9": float(np.mean(off > 0.9)) if off.size else float("nan"),
        "max_sim_to_main": float(qsim),
    }

def top_similar_facet_pairs(bp: dict, min_sim: float = 0.85, top_n: int = 10) -> pd.DataFrame:
    facets = [str(x) for x in (bp.get("facet_queries") or []) if str(x).strip()]
    if len(facets) < 2:
        return pd.DataFrame([])

    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vec.fit_transform(facets)
    sim = (X @ X.T).toarray()
    pairs = []
    for i in range(len(facets)):
        for j in range(i + 1, len(facets)):
            s = float(sim[i, j])
            if s >= float(min_sim):
                pairs.append({"i": i, "j": j, "sim": s, "facet_i": facets[i], "facet_j": facets[j]})
    out = pd.DataFrame(pairs).sort_values("sim", ascending=False).head(int(top_n)) if pairs else pd.DataFrame([])
    return out

rows = []
for cid, bp in base_blueprints.items():
    rows.append({"blueprint_source": "baseline", "chapter_id": cid, **facet_redundancy_stats(bp)})

facet_df = pd.DataFrame(rows).sort_values(["chapter_id"]).reset_index(drop=True)
display(facet_df)

# Show near-duplicate facets if any
for cid, bp in base_blueprints.items():
    pairs = top_similar_facet_pairs(bp, min_sim=0.85, top_n=8)
    if not pairs.empty:
        print("\nNear-duplicate facets (baseline) for", cid)
        display(pairs)

out_csv = STAGEB_DIR / "facet_redundancy_baseline.csv"
facet_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_facet_redundancy (RUN_MODE=sweeps)


In [19]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_coverage_proxy (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 3 (should-do): Downstream coverage proxy
# Using the already-fetched StageA corpus (built from baseline blueprint queries),
# compare how many labeled relevant items are retrieved by:
# - main query only
# - union of main + facet queries (top-K per query)

STAGEA_DIR = DATASET_PATH.parent / "stageA"
if not STAGEA_DIR.exists():
    # Backwards compatibility (older datasets)
    STAGEA_DIR = Path("eval_dataset/stageA")

TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)
TOPK_PER_QUERY = 250

def _clean_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()

def _build_text(title, abstract) -> str:
    t = _clean_text(title)
    a = _clean_text(abstract)
    if t and a:
        return f"{t}. {a}"
    return t or a

def _top_merge_keys(vec: TfidfVectorizer, X, df_docs: pd.DataFrame, query: str, topk: int) -> List[str]:
    q = vec.transform([str(query or "")])
    scores = (X @ q.T).toarray().ravel()
    topk = int(min(max(1, topk), len(scores)))
    idx = np.argpartition(-scores, topk - 1)[:topk]
    idx = idx[np.argsort(-scores[idx])]
    return df_docs.iloc[idx]["merge_key"].astype(str).tolist()

def _recall(rel: set, retrieved: set) -> float:
    return float(len(rel & retrieved) / len(rel)) if len(rel) > 0 else float("nan")

rows = []
for cid, bp in base_blueprints.items():
    stageA_path = STAGEA_DIR / f"stageA_combined_oa_s2_{cid}.csv"
    assert stageA_path.exists(), f"Missing {stageA_path}"

    docs = pd.read_csv(stageA_path)
    docs = docs.dropna(subset=["merge_key"]).copy()

    titles = docs["title"] if "title" in docs.columns else pd.Series([""] * len(docs))
    abstracts = docs["abstract"] if "abstract" in docs.columns else pd.Series([""] * len(docs))
    docs["_text"] = [_build_text(t, a) for t, a in zip(titles, abstracts)]

    # Relevant sets from labels (note: labels exist only for candidate subset)
    lab = df[df["chapter_id"].astype(str) == cid].copy()
    inc = set(lab[lab["final_label"] == "include"]["merge_key"].astype(str).tolist())
    incmaybe = set(lab[lab["final_label"].isin(["include", "maybe"])]["merge_key"].astype(str).tolist())

    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
    )
    X = vec.fit_transform(docs["_text"].astype(str).tolist())

    main_q = str(bp.get("main_query") or "")
    facet_qs = [str(x) for x in (bp.get("facet_queries") or [])]
    queries = [main_q] + facet_qs

    union_keys = set()
    for q in queries:
        union_keys.update(_top_merge_keys(vec, X, docs, q, topk=TOPK_PER_QUERY))
    union_budget = len(union_keys)

    main_keys_equal = set(_top_merge_keys(vec, X, docs, main_q, topk=union_budget)) if union_budget > 0 else set()

    rows.append({
        "stageB_run_id": stageB_run_id,
        "chapter_id": cid,
        "stageA_docs": int(len(docs)),
        "n_facets": int(len(facet_qs)),
        "topk_per_query": int(TOPK_PER_QUERY),
        "union_budget": int(union_budget),
        "n_include": int(len(inc)),
        "n_include_or_maybe": int(len(incmaybe)),
        "recall_include_main_equalbudget": _recall(inc, main_keys_equal),
        "recall_include_facets_union": _recall(inc, union_keys),
        "recall_incmaybe_main_equalbudget": _recall(incmaybe, main_keys_equal),
        "recall_incmaybe_facets_union": _recall(incmaybe, union_keys),
    })

coverage_df = pd.DataFrame(rows)
display(coverage_df)

out_csv = STAGEB_DIR / "coverage_proxy_baseline.csv"
coverage_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_coverage_proxy (RUN_MODE=sweeps)


In [20]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_prompt_variant_ab (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 4 (optional): A/B blueprint prompt variant (coverage_v1)
# NOTE: This evaluates blueprint-level quality metrics only.
# For a true end-to-end A/B, rebuild StageA+candidates+labels using the variant blueprint queries under a separate dataset tag.

variant_tag = "coverage_v1"
variant_dir = VARIANTS_DIR / variant_tag
variant_dir.mkdir(parents=True, exist_ok=True)

if not DO_STAGEB_API_CALLS:
    print("Skipping variant blueprint generation (DO_STAGEB_API_CALLS=False)")
elif not STAGEB_AGENTS_OK:
    print("Skipping variant blueprint generation (openai/agents import failed)")
else:
    assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
    _ = OpenAI()

    variant_agent = Agent(
        name=f"Chapter Blueprint Builder ({variant_tag})",
        model=BLUEPRINT_MODEL,
        model_settings=ModelSettings(top_p=1.0, verbosity="low"),
        instructions=VARIANT_COVERAGE_V1_INSTRUCTIONS,
        output_type=ChapterBlueprint,
    )

    sem = asyncio.Semaphore(int(STAGEB_CONCURRENCY))

    async def gen_variant(chapter: dict) -> dict:
        out_path = variant_dir / f"{chapter['chapter_id']}.json"
        if out_path.exists() and not STAGEB_FORCE_REGEN:
            return {"chapter_id": chapter["chapter_id"], "path": str(out_path), "cached": True, **{"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}}

        prompt = (
            "Create a ChapterBlueprint for academic literature retrieval.\n"
            "Return ONLY the structured output fields required by the schema.\n\n"
            "CHAPTER_SPEC_JSON:\n" + json.dumps(chapter, ensure_ascii=False, indent=2)
        )

        async with sem:
            res = await Runner.run(variant_agent, prompt)

        bp = res.final_output.model_dump()
        bp["_meta"] = {
            "chapter_id": chapter["chapter_id"],
            "variant": variant_tag,
            "model": BLUEPRINT_MODEL,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

        usage = res.context_wrapper.usage
        u = cost_from_usage(usage, model=BLUEPRINT_MODEL)
        return {"chapter_id": chapter["chapter_id"], "path": str(out_path), "cached": False, **u}

    t0 = time.time()
    rows = await asyncio.gather(*[gen_variant(ch) for ch in CHAPTERS])
    dt = time.time() - t0

    variant_runs = pd.DataFrame(rows)
    display(variant_runs)

    totals = {k: float(variant_runs[k].sum()) for k in ["requests", "input_tokens", "cached_input_tokens", "output_tokens", "cost_usd"] if k in variant_runs.columns}
    totals["cached_files"] = int((variant_runs.get("cached") == True).sum())
    totals["seconds"] = float(dt)
    print("Variant generation totals:", totals)

    out_csv = STAGEB_DIR / f"blueprint_variant_{variant_tag}_runs.csv"
    variant_runs.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

# Compare baseline vs variant on redundancy metrics (offline)
variant_blueprints = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    p = variant_dir / f"{cid}.json"
    if p.exists():
        variant_blueprints[cid] = json.loads(p.read_text(encoding="utf-8"))

rows = []
for cid, bp in base_blueprints.items():
    rows.append({"variant": "baseline", "chapter_id": cid, **facet_redundancy_stats(bp)})
for cid, bp in variant_blueprints.items():
    rows.append({"variant": variant_tag, "chapter_id": cid, **facet_redundancy_stats(bp)})

cmp_df = pd.DataFrame(rows).sort_values(["chapter_id", "variant"]).reset_index(drop=True)
display(cmp_df)

out_csv = STAGEB_DIR / f"facet_redundancy_{variant_tag}_compare.csv"
cmp_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("\nNOTE: True A/B requires re-running eval_dataset_builder.ipynb with the variant blueprints to fetch a new StageA corpus.")
    '''
    exec(_CODE, globals())


Skipping st_stageB_prompt_variant_ab (RUN_MODE=sweeps)


## Next steps (recommended)

1) **Freeze a baseline**: keep the `runs.csv` history and don’t change evaluation logic while tuning.
2) Make changes one-at-a-time (ablations), e.g.:
   - weights (`W_EMBED`, `W_TFIDF`, `CITE_WEIGHT`)
   - candidate pool size per facet (`TOP_PER_QUERY`)
   - candidate set size (`CAND_TARGET_N`)
3) After a few promising changes, re-run `eval_dataset_builder.ipynb` to regenerate candidates/labels and re-evaluate here.


In [21]:
if RUN_MODE != "stageB_ab":
    print(f"Skipping st_stageB_ab_compare (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage B end-to-end A/B compare (scientific workflow)
#
# Why this exists:
# - Stage B (blueprints) changes Stage A queries → fetched corpus → candidate pool.
# - So to compare Stage B variants scientifically, we must build separate datasets.
# Protocol:
# 1) Run `eval_dataset_builder.ipynb` twice (or more):
#    - Set `BLUEPRINT_VARIANT=baseline` (then run) → produces `eval_dataset/datasets/<tag>/labeled_dataset.csv`
#    - Set `BLUEPRINT_VARIANT=coverage_v1` (then run) → produces another dataset
#    - IMPORTANT: keep `LABEL_RUBRIC_SOURCE=eval_rubrics` so labels are comparable across datasets.
# 2) For each dataset, run the baseline evaluation cells in this notebook so it appends to `eval_dataset/experiments/runs.csv`.
#    - Easiest: set env var `DATASET_PATH` to the dataset's `.../labeled_dataset.csv` and re-run.
# 3) Finally run this cell to compare the macro metrics across datasets.

COMPARE_EXPERIMENT = STAGEB_AB_EXPERIMENT_TAG if RUN_MODE == "stageB_ab" else EXPERIMENT_TAG
COMPARE_SCORE_COL = STAGEB_AB_SCORE_COL if RUN_MODE == "stageB_ab" else "score_hybrid_pool"
DATASET_TAGS = STAGEB_AB_DATASET_TAGS if RUN_MODE == "stageB_ab" else []

runs_csv = EXP_DIR / "runs.csv"
if not runs_csv.exists():
    raise FileNotFoundError(f"Missing runs.csv at {runs_csv}. Run at least one evaluation first.")

runs = pd.read_csv(runs_csv)
runs = runs.copy()

if "experiment" not in runs.columns:
    raise ValueError("runs.csv schema is unexpected (missing 'experiment' column)")
if "score_col" not in runs.columns:
    raise ValueError("runs.csv schema is unexpected (missing 'score_col' column)")

runs = runs[(runs["experiment"].astype(str) == str(COMPARE_EXPERIMENT)) & (runs["score_col"].astype(str) == str(COMPARE_SCORE_COL))]
if runs.empty:
    raise ValueError(
        "No rows matched (experiment, score_col). "
        "Did you run an evaluation that appended to runs.csv with the same EXPERIMENT_TAG and score_col?"
    )


def dataset_tag_from_path(p: str) -> str:
    s = ("" if p is None else str(p)).replace("\\", "/")
    needle = "eval_dataset/datasets/"
    if needle not in s:
        return "legacy"
    rest = s.split(needle, 1)[1]
    return rest.split("/", 1)[0]


runs["dataset_tag"] = runs["dataset_path"].apply(dataset_tag_from_path)
runs["created_at_utc"] = pd.to_datetime(runs.get("created_at_utc"), utc=True, errors="coerce")

if DATASET_TAGS:
    runs = runs[runs["dataset_tag"].isin(DATASET_TAGS)]

# Auto selection: compare the most recent datasets found in runs.csv
if not DATASET_TAGS:
    most_recent = (
        runs.dropna(subset=["created_at_utc"])
        .sort_values("created_at_utc", ascending=False)
        .drop_duplicates(subset=["dataset_tag"])
        .head(6)
    )
    DATASET_TAGS = list(most_recent["dataset_tag"].astype(str).values)
    print("Auto-selected dataset tags:", DATASET_TAGS)
    runs = runs[runs["dataset_tag"].isin(DATASET_TAGS)]

# Latest macro row per dataset_tag
latest = (
    runs.sort_values("created_at_utc", ascending=True)
    .groupby(["dataset_tag"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# Join dataset manifests (if present)
datasets_root = Path("eval_dataset/datasets")
manifest_rows = []
for tag in latest["dataset_tag"].astype(str).unique():
    if tag == "legacy":
        manifest_rows.append({"dataset_tag": tag})
        continue
    mp = datasets_root / tag / "manifest.json"
    if not mp.exists():
        manifest_rows.append({"dataset_tag": tag})
        continue
    try:
        m = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        m = {}

    manifest_rows.append(
        {
            "dataset_tag": tag,
            "blueprint_variant": m.get("blueprint_variant"),
            "label_rubric_source": m.get("label_rubric_source"),
            "dataset_created_at_utc": m.get("created_at_utc"),
        }
    )

manifest_df = pd.DataFrame(manifest_rows).drop_duplicates(subset=["dataset_tag"], keep="first")
summary = latest.merge(manifest_df, on="dataset_tag", how="left")

cols = [
    "dataset_tag",
    "blueprint_variant",
    "label_rubric_source",
    "dataset_sha1_12",
    "created_at_utc",
    "ndcg@20",
    "p@20",
    "mrr_include",
    "auc_include",
    "n_docs",
    "n_include",
    "n_maybe",
    "dataset_path",
    "run_id",
]
cols = [c for c in cols if c in summary.columns]
summary = summary[cols].sort_values("ndcg@20", ascending=False).reset_index(drop=True)

display(summary)

# Optional: metric deltas if exactly two datasets are specified
if DATASET_TAGS and len(DATASET_TAGS) == 2 and {"ndcg@20", "p@20", "mrr_include", "auc_include"}.issubset(summary.columns):
    a, b = DATASET_TAGS
    s = summary.set_index("dataset_tag")
    if a in s.index and b in s.index:
        delta = (s.loc[b, ["ndcg@20", "p@20", "mrr_include", "auc_include"]] - s.loc[a, ["ndcg@20", "p@20", "mrr_include", "auc_include"]]).to_frame("delta")
        print(f"\nDelta ({b} - {a}):")
        display(delta)

# Persist comparison output
out_path = EXP_DIR / f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_stageB_ab_compare.csv"
summary.to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageB_ab_compare (RUN_MODE=sweeps)
